# CareLoop Clinics — Part 1: Ingest, Validate, Clean and Enrich
This notebook completes Tasks 1.1–1.6 and writes the required hand-off files.

In [ ]:
import csv, json, requests, time
import pandas as pd
import numpy as np
from pathlib import Path
DATA = Path("data")

## Task 1.1 — Read with the `csv` module

In [ ]:
with open(DATA / "appointments.csv", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    header = reader.fieldnames
    rows = list(reader)
print("Header:", header)
print("Row count:", len(rows))

def parse_row(row):
    converted = dict(row)
    converted["age"] = int(float(converted["age"]))
    converted["fee_inr"] = int(float(converted["fee_inr"]))
    converted["prior_no_shows"] = int(float(converted["prior_no_shows"]))
    converted["distance_km"] = float(converted["distance_km"])
    return converted

accepted, rejected = [], []
for row in rows:
    try:
        accepted.append(parse_row(row))
    except ValueError:
        rejected.append(row)
print("Accepted:", len(accepted), "Rejected:", len(rejected))
with open("rejected_rows.json", "w", encoding="utf-8") as f:
    json.dump(rejected, f, indent=2)

## Tasks 1.2 and 1.3 — Inspect and clean

In [ ]:
df = pd.read_csv(DATA / "appointments.csv")
print("Raw shape:", df.shape)
df.info()
print("\nNulls:\n", df.isnull().sum())
print("Duplicates:", df.duplicated().sum())
print("Raw booking-channel spellings:", df["booking_channel"].nunique())

df = df.drop_duplicates().copy(); print("After duplicate removal:", len(df))
df["booking_channel"] = df["booking_channel"].str.strip().str.title()
print("Clean booking-channel labels:", sorted(df["booking_channel"].dropna().unique()))
df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")
df["booked_on"] = pd.to_datetime(df["booked_on"])
print("Unreadable/missing appointment dates:", df["appt_date"].isna().sum())
df = df.dropna(subset=["appt_date", "status"]).copy(); print("After required-field removal:", len(df))

invalid_age = (df["age"] < 1) | (df["age"] > 100)
print("Invalid ages:", invalid_age.sum())
df.loc[invalid_age, "age"] = np.nan
age_median = df["age"].median(); distance_median = df["distance_km"].median()
print("Age median:", age_median, "Distance median:", distance_median)
df["age"] = df["age"].fillna(age_median).round().astype(int)
df["distance_km"] = df["distance_km"].fillna(distance_median)

for column in ["distance_km", "fee_inr"]:
    q1 = df[column].quantile(0.25); q3 = df[column].quantile(0.75); iqr = q3 - q1
    lower = q1 - 1.5 * iqr; upper = q3 + 1.5 * iqr
    capped = ((df[column] < lower) | (df[column] > upper)).sum()
    print(column, "Q1:", q1, "Q3:", q3, "lower:", lower, "upper:", upper, "capped:", capped)
    df[column] = np.clip(df[column], lower, upper)
df["distance_km"] = df["distance_km"].round(1)
df["fee_inr"] = df["fee_inr"].round().astype(int)
print("Largest cleaned fee:", df["fee_inr"].max())

## Task 1.4 — Features and NumPy summary

In [ ]:
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days
df["no_show"] = (df["status"] == "No-Show").astype(int)
X_np = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()
print("Lead-day range:", df["lead_days"].min(), df["lead_days"].max())
print("No-shows:", df["no_show"].sum(), "Rate:", round(100 * df["no_show"].mean(), 1), "%")
print("Column means:", X_np.mean(axis=0))
mask = X_np[:, 3] > 14
print("Lead days > 14:", mask.sum(), round(100 * mask.mean(), 1), "%")
distance = X_np[:, 1]
z = (distance - np.mean(distance)) / np.std(distance)
print("Distance |z| > 2:", (np.abs(z) > 2).sum())

## Task 1.5 — Open-Meteo enrichment

In [ ]:
clinics = pd.read_csv(DATA / "clinics.csv")
temperatures = []
for _, clinic in clinics.iterrows():
    temp = None
    reason = None
    for attempt in range(2):
        try:
            response = requests.get(
                "https://api.open-meteo.com/v1/forecast",
                params={"latitude": clinic["latitude"], "longitude": clinic["longitude"],
                        "hourly": "temperature_2m", "forecast_days": 1},
                timeout=10)
            if response.status_code != 200:
                reason = f"HTTP {response.status_code}"
                continue
            payload = response.json()
            hourly = payload.get("hourly", {}).get("temperature_2m")
            if hourly:
                temp = hourly[0]
                break
            reason = "hourly temperature missing"
        except requests.exceptions.RequestException as exc:
            reason = str(exc)
        if attempt == 0:
            time.sleep(1)
    if temp is None:
        print(clinic["clinic_name"], "failed:", reason)
    temperatures.append(temp)
clinics["temp_c"] = temperatures
clinics.to_csv("clinics_enriched.csv", index=False)
print(clinics)

## Task 1.6 — Final exports and SQL load script

In [ ]:
columns = ["appointment_id","patient_id","clinic_id","booked_on","appt_date","age","gender",
           "distance_km","fee_inr","booking_channel","visit_type","reminder_sent",
           "prior_no_shows","status","lead_days","no_show"]
clean = df.sort_values("appointment_id").copy()
clean["booked_on"] = clean["booked_on"].dt.strftime("%Y-%m-%d")
clean["appt_date"] = clean["appt_date"].dt.strftime("%Y-%m-%d")
clean = clean[columns]
print("Appointment IDs unique:", clean["appointment_id"].is_unique)
print("Missing values:", int(clean.isna().sum().sum()))
print("Final shape:", clean.shape)
clean.to_csv("appointments_clean.csv", index=False)

def quote(value): return "'" + str(value) + "'"
def number(value):
    if pd.isna(value): return "NULL"
    if isinstance(value, float) and value.is_integer(): return str(int(value))
    return str(value)
text_columns = {"appointment_id","patient_id","clinic_id","booked_on","appt_date","gender",
                "booking_channel","visit_type","reminder_sent","status"}
with open("load_data.sql", "w", encoding="utf-8") as f:
    for _, row in clinics.iterrows():
        values = [quote(row["clinic_id"]), quote(row["clinic_name"]), quote(row["city"]),
                  quote(row["tier"]), number(row["latitude"]), number(row["longitude"]), number(row["temp_c"])]
        f.write("INSERT INTO clinics VALUES (" + ",".join(values) + ");\n")
    for _, row in clean.iterrows():
        values = [quote(row[c]) if c in text_columns else number(row[c]) for c in columns]
        f.write("INSERT INTO appointments VALUES (" + ",".join(values) + ");\n")
print("INSERT statements:", sum(1 for _ in open("load_data.sql", encoding="utf-8")))